This notebook is an end-to-end use case showing how Pixeltable treats video, audio, images, and text as first-class data types — storage, orchestration, and retrieval in one table interface.

> **Want to build a full app on top of this?** See the [Pixeltable Starter Kit](https://github.com/pixeltable/pixeltable-starter-kit) (FastAPI + React).

**What it does:** Turn a source video into targeted ad variants for different audiences — automatically.

## Problem

You have source videos and need short ad variants for different audiences. Manually editing each variant is slow and expensive.

| Use case | Input | Output |
|----------|-------|--------|
| Real estate | Walkthrough + buyer persona | Targeted listing ads |
| Automotive | Car review + region | Regional dealer ads |
| Hospitality | Hotel tour + season | Seasonal promo clips |
| E-commerce | Product demo + demographic | Audience-specific ads |

## Solution

**What Pixeltable handles:**

- **Storage + orchestration + retrieval in one table.** Videos, images, AI responses, and metadata live side-by-side.
- **Incremental processing.** New rows flow through the pipeline automatically. New columns backfill.
- **Caching and retries.** API calls are rate-limited, cached, and retried on failure.
- **Versioned by default.** Roll back or time-travel to any prior state.
- **Experiment, don’t plumb.** Test any transformation with `.select().head()` before committing it as a column.

```
                   ┌────────────────────────────────────────────────────────────┐
                   │         UNDERSTAND            GENERATE          DELIVER  │
                   │                                                          │
               ┌──▶ Audio ─▶ Whisper ──────┐                              │
               │   │       (local, free)     │                              │
 Video ───────┤   │                          ▼                              │
               │   │                    Ad Script ──▶ Hero Image            │
 Property ────├──▶ Gemini Multimodal ────┘   (Gemini)       (Imagen)             │
   Type        │   │       (API)                  │           │              │
               │   │                               │    first  │              │
 Target  ─────┘   │                               │    frame  ▼              │
 Audience          │                               │       Ad Video           │
                   │                               └─────▶ (Veo 3)            │
                   │                                          │              │
                   │                                          ▼              │
                   │                                    Captioned Ad       │
                   │                                    (text overlay)      │
                   │                                          │              │
                   │                                          ▼              │
                   │                                    Final Video ──▶ URL │
                   │                                    (Tigris S3)        │
                   └────────────────────────────────────────────────────────────┘
```

Whisper gives exact words. Gemini gives visual context. The ad script combines both.

| Step | Model | Purpose |
|------|-------|---------|
| Transcribe | Whisper (local) | Verbatim transcript |
| Analyze | Gemini 2.5 Flash | Visual + contextual understanding |
| Script | Gemini 2.5 Flash | Audience-targeted ad hook |
| Image | Imagen 4.0 | Hero visual |
| Video | Veo 3 | Animated clip from hero image |
| Captions | Built-in | Text overlay |
| Store | Tigris S3 (optional) | Presigned URLs |

**API keys needed:** Gemini only. Whisper runs locally. Tigris is optional.

### Setup

In [ ]:
%pip install -qU pixeltable google-genai openai-whisper

In [ ]:
import os
import getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API Key: ')

# Optional: Tigris storage for sharing generated videos via URLs
if 'AWS_ACCESS_KEY_ID' not in os.environ:
    os.environ['AWS_ACCESS_KEY_ID'] = getpass.getpass('Tigris Access Key ID (or press Enter to skip): ')
    if os.environ['AWS_ACCESS_KEY_ID']:
        os.environ['AWS_SECRET_ACCESS_KEY'] = getpass.getpass('Tigris Secret Access Key: ')
        bucket_name = getpass.getpass('Tigris Bucket Name: ')
        os.environ['AWS_ENDPOINT_URL_S3'] = 'https://t3.storage.dev'
        os.environ['AWS_REGION'] = 'auto'
    else:
        bucket_name = None
else:
    bucket_name = os.environ.get('TIGRIS_BUCKET', 'pixeltable-video-ads')

In [ ]:
import re

import pixeltable as pxt
from google.genai.types import GenerateImagesConfigDict
from pixeltable.functions import gemini, whisper

In [ ]:
pxt.drop_dir('video_ads', force=True)
pxt.create_dir('video_ads')

## Build the Ad Pipeline

### Step 1: Source Videos Table

In [ ]:
ads = pxt.create_table(
    'video_ads.ads',
    {
        'video': pxt.Video,
        'property_type': pxt.String,
        'target_audience': pxt.String,
    }
)

### Step 2: Understand the Video (Two Parallel Paths)

- **Whisper** (local): exact verbatim transcript
- **Gemini multimodal** (API): visual scenes, mood, production quality

Both run as independent computed columns and combine downstream.

In [ ]:
@pxt.udf
def video_analysis_prompt(video: pxt.Video, property_type: str) -> pxt.Json:
    """Pair the video file with an analysis prompt for Gemini multimodal."""
    return [
        video,
        f'Analyze this {property_type} video. Describe the key visuals, mood, '
        'production quality, and selling points. Be specific and concise.'
    ]

In [ ]:
# Path A: Local Whisper transcription (exact verbatim text)
ads.add_computed_column(audio=ads.video.extract_audio(format='wav'))
ads.add_computed_column(transcription=whisper.transcribe(ads.audio, model='base'))
ads.add_computed_column(transcript=ads.transcription['text'].astype(pxt.String))

# Path B: Gemini multimodal video understanding (visual + contextual analysis)
ads.add_computed_column(
    analysis_contents=video_analysis_prompt(ads.video, ads.property_type)
)
ads.add_computed_column(
    video_analysis=gemini.generate_content(
        ads.analysis_contents,
        model='gemini-2.5-flash'
    )
)
ads.add_computed_column(
    analysis_text=ads.video_analysis['candidates'][0]['content']['parts'][0]['text'].astype(pxt.String)
)

### Step 3: Generate Ad Script

Gemini writes the ad script using both the Whisper transcript and the Gemini visual analysis.

In [ ]:
@pxt.udf
def ad_script_prompt(
    transcript: str, analysis: str, property_type: str, target_audience: str
) -> str:
    """Combine transcript + visual analysis into a script prompt."""
    return (
        f'Write a 2-3 sentence video ad hook (under 15 seconds spoken) for a '
        f'{property_type} listing targeting {target_audience}.\n\n'
        f'TRANSCRIPT: {transcript[:600]}\n\n'
        f'VISUAL ANALYSIS: {analysis[:600]}\n\n'
        'Use details from both. Conversational and specific. '
        'Return ONLY the script text.'
    )

In [ ]:
ads.add_computed_column(
    script_prompt=ad_script_prompt(
        ads.transcript, ads.analysis_text, ads.property_type, ads.target_audience
    )
)

ads.add_computed_column(
    script_response=gemini.generate_content(
        ads.script_prompt,
        model='gemini-2.5-flash'
    )
)

ads.add_computed_column(
    ad_script=ads.script_response['candidates'][0]['content']['parts'][0]['text'].astype(pxt.String)
)

### Step 4: Generate Hero Image with Imagen

In [ ]:
@pxt.udf
def image_prompt(ad_script: str, property_type: str) -> str:
    """Build prompt for Imagen hero image."""
    return (
        f'Cinematic vertical photo of a {property_type}, golden-hour lighting, '
        f'evoking: "{ad_script[:200]}". No text or watermarks.'
    )

In [ ]:
ads.add_computed_column(
    image_prompt=image_prompt(ads.ad_script, ads.property_type)
)

ads.add_computed_column(
    hero_image=gemini.generate_images(
        ads.image_prompt,
        model='imagen-4.0-fast-generate-001',
        config=GenerateImagesConfigDict(aspect_ratio='9:16')
    )
)

### Step 5: Generate Ad Video with Veo 3

Visual-only prompt (no baked-in narration) so we control captions separately.

In [ ]:
VIDEO_PROMPT = (
    'Subtle cinematic camera movement, professional real estate footage, '
    'warm lighting, smooth motion. No text, no narration, no speech.'
)

ads.add_computed_column(
    ad_video=gemini.generate_videos(
        prompt=VIDEO_PROMPT,
        image=ads.hero_image,
        model='veo-3.0-generate-001'
    )
)

### Step 6: Add Captions

In [ ]:
@pxt.udf
def clean_caption(text: str) -> str:
    """Strip markdown artifacts and truncate for on-screen display."""
    clean = re.sub(r'[*_#"\'\n]+', ' ', text).strip()
    clean = re.sub(r' {2,}', ' ', clean)
    return clean[:57].rsplit(' ', 1)[0] + '...' if len(clean) > 60 else clean


ads.add_computed_column(caption=clean_caption(ads.ad_script))

ads.add_computed_column(
    captioned_ad=ads.ad_video.overlay_text(
        ads.caption,
        font_size=28,
        color='white',
        box=True,
        box_color='black',
        box_opacity=0.7,
        box_border=[8, 16],
        horizontal_align='center',
        vertical_align='bottom',
        vertical_margin=80
    )
)

### Step 7: Store for Sharing (Optional)

In [ ]:
if bucket_name:
    ads.add_computed_column(
        final_video=ads.captioned_ad,
        destination=f's3://{bucket_name}/video-ads/'
    )

## Demo: Generate a Property Ad

In [ ]:
sample_video_url = 'https://github.com/pixeltable/pixeltable/raw/release/docs/resources/audio-transcription-demo/Lex-Fridman-Podcast-430-Excerpt-0.mp4'

ads.insert([
    {
        'video': sample_video_url,
        'property_type': 'luxury condo',
        'target_audience': 'luxury seekers'
    }
])

In [ ]:
ads.select(
    ads.property_type,
    ads.target_audience,
    ads.transcript,
    ads.analysis_text
).head()

In [ ]:
ads.select(
    ads.target_audience,
    ads.ad_script,
    ads.hero_image
).head()

In [ ]:
ads.select(
    ads.target_audience,
    ads.captioned_ad
).head()

### Share via Presigned URLs

In [ ]:
if bucket_name:
    from pixeltable.functions import net

    ads.select(
        ads.target_audience,
        ads.ad_script,
        shareable_url=net.presigned_url(ads.final_video.fileurl, 3600)
    ).head()

## Scale: Multiple Audiences from One Video

Same video, different audiences — each gets a unique ad.

In [ ]:
ads.insert([
    {'video': sample_video_url, 'property_type': 'luxury condo', 'target_audience': 'first-time buyers'},
    {'video': sample_video_url, 'property_type': 'luxury condo', 'target_audience': 'investors'},
    {'video': sample_video_url, 'property_type': 'luxury condo', 'target_audience': 'relocators'},
])

In [ ]:
ads.select(
    ads.target_audience,
    ads.ad_script,
    ads.hero_image
).collect()

## Pipeline Overview

In [ ]:
ads.describe()

## Customize This Pipeline

Pixeltable is the infrastructure. **You** decide what goes in the pipeline. Swap one step and everything downstream recomputes.

### Swap Any Model

```python
from pixeltable.functions import openai

ads.add_computed_column(
    script_response=openai.chat_completions(
        messages=[{'role': 'user', 'content': ads.script_prompt}],
        model='gpt-4o'
    )
)
```

```python
from pixeltable.functions import fal

ads.add_computed_column(
    ad_video=fal.run('fal-ai/luma-dream-machine/image-to-video', ...)
)
```

### Use Real Footage Instead of AI Visuals

```python
from pixeltable.functions.video import extract_frame

ads.add_computed_column(hero_image=extract_frame(ads.video, 5.0))
```

### Add TTS for Exact Narration

Add a TTS step and overlay the audio onto the video:

```python
from pixeltable.functions import huggingface
from pixeltable.functions.video import with_audio

ads.add_computed_column(
    narration=huggingface.text_to_speech(ads.ad_script, model_id='microsoft/speecht5_tts')
)
ads.add_computed_column(ad_with_narration=with_audio(ads.ad_video, ads.narration))
```

### Adapt for Any Industry

```python
ads.insert([{'video': car_review_url, 'property_type': 'SUV', 'target_audience': 'families'}])
ads.insert([{'video': hotel_tour_url, 'property_type': 'resort', 'target_audience': 'honeymooners'}])
```

### Related Notebooks

- [Generate Videos with AI](../cookbooks/video/video-generate-ai.ipynb)
- [Video Thumbnails](../cookbooks/video/video-generate-thumbnails.ipynb)
- [Working with Gemini](../providers/working-with-gemini.ipynb)
- [Working with Tigris](../providers/working-with-tigris.ipynb)